In [5]:
# ✅ (Recommended) Force single GPU + reduce CUDA fragmentation
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [6]:
# ✅ Install deps (Kaggle)
!pip -q install -U transformers datasets accelerate evaluate iterative-stratification


In [7]:
# ✅ Imports
import os, re, random, math, glob, json, shutil
import numpy as np
import pandas as pd
import torch
from torch import nn

from datasets import Dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)


In [8]:
# ✅ Config (T4 16GB OOM-safe for xlm-roberta-large)
# Auto-detect CSV from /kaggle/input
csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print("✅ CSVs found:")
for i,p in enumerate(csvs):
    print(f"[{i}] {p}")
if len(csvs)==0:
    raise FileNotFoundError("No CSV found in /kaggle/input. Add dataset via Kaggle → Add Input")

CSV_INDEX = 0
CSV_PATH = csvs[CSV_INDEX]

TEXT_COL = "Data"
LABEL_COLS = ["Love", "Joy", "Anger", "Surprise", "Sadness", "Fear", "Hate"]

MODEL_NAME = "xlm-roberta-large"   # if still OOM, use "xlm-roberta-base"
MAX_LEN = 160                      # if OOM, use 128

BATCH_SIZE = 4                     # OOM-safe on T4 for xlm-roberta-large
GRAD_ACCUM = 8                     # effective batch = 32
EPOCHS = 6
LR = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

OUTPUT_DIR = "/kaggle/working/banglish-emotion-xlmr-large"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n✅ Using CSV_PATH:", CSV_PATH)


✅ CSVs found:
[0] /kaggle/input/model-dataset/Model_dataset.csv

✅ Using CSV_PATH: /kaggle/input/model-dataset/Model_dataset.csv


In [9]:
# ✅ Load + clean dataset
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.astype(str).str.strip()

# Auto-fix text col if needed
if TEXT_COL not in df.columns:
    candidates = ["text","Text","data","Data","sentence","Sentence","content","Content"]
    found = next((c for c in candidates if c in df.columns), None)
    if found is None:
        raise ValueError(f"TEXT_COL '{TEXT_COL}' not found. Available columns: {df.columns.tolist()}")
    print(f"⚠️ TEXT_COL '{TEXT_COL}' not found. Using '{found}' instead.")
    TEXT_COL = found

missing_labels = [c for c in LABEL_COLS if c not in df.columns]
if missing_labels:
    raise ValueError(f"Missing label columns: {missing_labels}\nAvailable columns: {df.columns.tolist()}")

df = df[[TEXT_COL] + LABEL_COLS].copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

for c in LABEL_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int).clip(0,1)

print("✅ Samples:", len(df))
print("✅ Label distribution:\n", df[LABEL_COLS].sum().sort_values(ascending=False))
df.head()


✅ Samples: 27809
✅ Label distribution:
 Love        4331
Joy         4331
Anger       4331
Surprise    4331
Sadness     4331
Fear        4331
Hate        4331
dtype: int64


,Data,Love,Joy,Anger,Surprise,Sadness,Fear,Hate
0,atae best story amar darun lagacha wow smiling...,0,0,0,1,0,0,0
1,onek valo hoto jodi phone ta patam,0,0,0,0,1,0,0
2,apu tmr moto dj hoite cai help me plz amake sh...,1,0,0,0,0,0,0
3,aidi bolte kivabe vromner khetre sotti kom bho...,0,0,0,0,0,1,0
4,ei chele ekta kulanggar jara nijer bapke shosh...,0,0,0,0,0,0,1


In [10]:
# ✅ Multilabel stratified split (best for multilabel)
X = df[TEXT_COL].values
Y = df[LABEL_COLS].values

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(mskf.split(X, Y))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df))


Train: 22247 Val: 5562


In [11]:
# ✅ Tokenizer + HF Datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=[TEXT_COL])
val_ds   = val_ds.map(tokenize_batch, batched=True, remove_columns=[TEXT_COL])

def add_labels(batch):
    labels = np.stack([batch[c] for c in LABEL_COLS], axis=1).astype(np.float32)
    return {"labels": labels}

train_ds = train_ds.map(add_labels, batched=True, remove_columns=LABEL_COLS)
val_ds   = val_ds.map(add_labels, batched=True, remove_columns=LABEL_COLS)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/22247 [00:00<?, ? examples/s]

Map:   0%|          | 0/5562 [00:00<?, ? examples/s]

Map:   0%|          | 0/22247 [00:00<?, ? examples/s]

Map:   0%|          | 0/5562 [00:00<?, ? examples/s]

In [12]:
# ✅ Model + memory savers (important for xlm-roberta-large on T4)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLS),
    problem_type="multi_label_classification"
)

# Big memory saver
model.gradient_checkpointing_enable()
model.config.use_cache = False

# ✅ GPU check (after model loaded)
print("GPU:", torch.cuda.get_device_name(0))
print("Allocated(GB):", round(torch.cuda.memory_allocated()/1e9, 2))
print("Reserved (GB):", round(torch.cuda.memory_reserved()/1e9, 2))


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPU: Tesla P100-PCIE-16GB
Allocated(GB): 0.0
Reserved (GB): 0.0


In [13]:
# ✅ pos_weight for imbalance (BCEWithLogitsLoss)
pos = train_df[LABEL_COLS].sum().values.astype(np.float32)
neg = (len(train_df) - pos).astype(np.float32)
pos_weight = torch.tensor(neg / (pos + 1e-6), dtype=torch.float32)
print("pos_weight:", pos_weight)


pos_weight: tensor([5.4205, 5.4205, 5.4205, 5.4205, 5.4205, 5.4205, 5.4205])


In [14]:
# ✅ Custom Trainer (DataParallel safe: NO model.device)
class MultilabelTrainer(Trainer):
    def __init__(self, pos_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        labels = labels.to(logits.device)
        pw = self.pos_weight.to(logits.device) if self.pos_weight is not None else None

        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pw)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics_with_threshold(eval_pred, threshold=0.5, per_label_thresholds=None):
    logits, labels = eval_pred
    probs = sigmoid_np(logits)

    if per_label_thresholds is not None:
        thr = np.array(per_label_thresholds).reshape(1, -1)
        preds = (probs >= thr).astype(int)
    else:
        preds = (probs >= threshold).astype(int)

    y_true = labels.astype(int)
    y_pred = preds.astype(int)

    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    p_micro  = precision_score(y_true, y_pred, average="micro", zero_division=0)
    r_micro  = recall_score(y_true, y_pred, average="micro", zero_division=0)
    subset_acc = accuracy_score(y_true, y_pred)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "precision_micro": p_micro,
        "recall_micro": r_micro,
        "subset_accuracy": subset_acc
    }


In [15]:
# ✅ TrainingArguments + Trainer (transformers version-safe: eval_strategy)
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
EVAL_SAVE_STEPS = max(steps_per_epoch, 200)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    eval_strategy="steps",
    eval_steps=EVAL_SAVE_STEPS,
    save_strategy="steps",
    save_steps=EVAL_SAVE_STEPS,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=max(EVAL_SAVE_STEPS // 5, 50),

    report_to="none",

    fp16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,

    gradient_accumulation_steps=GRAD_ACCUM,
    max_grad_norm=1.0,

    lr_scheduler_type="cosine",
    seed=42
)

print("✅ steps_per_epoch =", steps_per_epoch)
print("✅ eval/save every", EVAL_SAVE_STEPS, "steps")

def compute_metrics(eval_pred):
    return compute_metrics_with_threshold(eval_pred, threshold=0.5)

trainer = MultilabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,  # ✅ replaces tokenizer= (deprecated)
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    pos_weight=pos_weight
)


✅ steps_per_epoch = 696
✅ eval/save every 696 steps


In [16]:
# ✅ Train (resume if checkpoint exists)
torch.cuda.empty_cache()

last_ckpt = None
if os.path.isdir(OUTPUT_DIR):
    ckpts = []
    for d in os.listdir(OUTPUT_DIR):
        p = os.path.join(OUTPUT_DIR, d)
        if os.path.isdir(p) and d.startswith("checkpoint-"):
            try:
                step = int(d.split("-")[-1])
                ckpts.append((step, p))
            except:
                pass
    if len(ckpts) > 0:
        ckpts = sorted(ckpts, key=lambda x: x[0])
        last_ckpt = ckpts[-1][1]
        print("🔄 Resuming from:", last_ckpt)
    else:
        print("✅ No checkpoint found. Training from scratch.")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)

print("\n✅ Training finished.")
print("✅ Best checkpoint:", getattr(trainer.state, "best_model_checkpoint", None))

metrics = trainer.evaluate()
print("\n📊 Validation metrics (threshold=0.5):")
for k, v in metrics.items():
    if isinstance(v, (float, int)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")


✅ No checkpoint found. Training from scratch.


Step,Training Loss,Validation Loss,F1 Micro,F1 Macro,Precision Micro,Recall Micro,Subset Accuracy
696,1.017700,0.980105,0.378845,0.416710,0.250802,0.774002,0.023553
1392,0.751700,0.701975,0.557494,0.576870,0.416946,0.840977,0.147609
2088,0.626200,0.631172,0.605519,0.616664,0.468942,0.854339,0.243617
2784,0.539500,0.599598,0.625410,0.633937,0.494626,0.850214,0.307623
3480,0.456100,0.609369,0.638231,0.646003,0.514439,0.840482,0.336749
4176,0.422800,0.613708,0.644608,0.653088,0.524060,0.837182,0.348256



✅ Training finished.
✅ Best checkpoint: /kaggle/working/banglish-emotion-xlmr-large/checkpoint-4176



📊 Validation metrics (threshold=0.5):
eval_loss: 0.6137
eval_f1_micro: 0.6446
eval_f1_macro: 0.6531
eval_precision_micro: 0.5241
eval_recall_micro: 0.8372
eval_subset_accuracy: 0.3483
eval_runtime: 45.2565
eval_samples_per_second: 122.8990
eval_steps_per_second: 30.7360
epoch: 6.0000


In [17]:
# ✅ Threshold tuning on validation (Boost F1)
def find_best_thresholds_fast(ds, step=0.05):
    out = trainer.predict(ds)
    logits = out.predictions
    y_true = out.label_ids.astype(int)
    probs = sigmoid_np(logits)

    thresholds = np.arange(step, 1.0, step)

    # global threshold for micro-F1
    best_global_t, best_micro = 0.5, -1
    for t in thresholds:
        y_pred = (probs >= t).astype(int)
        micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
        if micro > best_micro:
            best_micro, best_global_t = micro, t

    # per-label thresholds
    best_per_label = []
    for j in range(y_true.shape[1]):
        best_tj, best_f1j = 0.5, -1
        for t in thresholds:
            y_pred_j = (probs[:, j] >= t).astype(int)
            f1j = f1_score(y_true[:, j], y_pred_j, zero_division=0)
            if f1j > best_f1j:
                best_f1j, best_tj = f1j, t
        best_per_label.append(best_tj)

    return float(best_global_t), [float(x) for x in best_per_label]

best_global_t, best_per_label_t = find_best_thresholds_fast(val_ds, step=0.05)

print("\n✅ Best global threshold (micro-F1):", best_global_t)
print("✅ Best per-label thresholds:", dict(zip(LABEL_COLS, best_per_label_t)))

preds = trainer.predict(val_ds)
metrics_global = compute_metrics_with_threshold((preds.predictions, preds.label_ids), threshold=best_global_t)
metrics_per_label = compute_metrics_with_threshold((preds.predictions, preds.label_ids), per_label_thresholds=best_per_label_t)

print("\n===== 📌 Validation Metrics (Global threshold) =====")
for k, v in metrics_global.items():
    print(f"{k}: {v:.4f}")

print("\n===== 📌 Validation Metrics (Per-label thresholds) =====")
for k, v in metrics_per_label.items():
    print(f"{k}: {v:.4f}")



✅ Best global threshold (micro-F1): 0.7500000000000001
✅ Best per-label thresholds: {'Love': 0.7000000000000001, 'Joy': 0.7000000000000001, 'Anger': 0.7000000000000001, 'Surprise': 0.7500000000000001, 'Sadness': 0.7000000000000001, 'Fear': 0.6500000000000001, 'Hate': 0.7500000000000001}



===== 📌 Validation Metrics (Global threshold) =====
f1_micro: 0.6701
f1_macro: 0.6738
precision_micro: 0.6255
recall_micro: 0.7215
subset_accuracy: 0.5083

===== 📌 Validation Metrics (Per-label thresholds) =====
f1_micro: 0.6714
f1_macro: 0.6767
precision_micro: 0.6060
recall_micro: 0.7527
subset_accuracy: 0.4802


In [18]:
# ✅ Save model + tokenizer + thresholds + ZIP
SAVE_PATH = "/kaggle/working/final_model_xlmr_large"
os.makedirs(SAVE_PATH, exist_ok=True)

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

thresholds = {
    "best_global_threshold": float(best_global_t),
    "best_per_label_thresholds": {lbl: float(t) for lbl, t in zip(LABEL_COLS, best_per_label_t)},
    "label_cols": LABEL_COLS,
    "text_col": TEXT_COL,
    "model_name": MODEL_NAME,
    "max_len": int(MAX_LEN)
}
with open(os.path.join(SAVE_PATH, "thresholds.json"), "w", encoding="utf-8") as f:
    json.dump(thresholds, f, ensure_ascii=False, indent=2)

zip_path = shutil.make_archive(SAVE_PATH, "zip", SAVE_PATH)
print("✅ Saved to:", SAVE_PATH)
print("✅ Zipped:", zip_path)


✅ Saved to: /kaggle/working/final_model_xlmr_large
✅ Zipped: /kaggle/working/final_model_xlmr_large.zip


In [19]:
# =========================
# ✅ Overall (Label-wise) Accuracy (NOT Subset Accuracy)
# Works for multi-label classification
# =========================
import numpy as np

def _sigmoid_np(x):
    x = np.asarray(x, dtype=np.float32)
    return 1.0 / (1.0 + np.exp(-x))

# ---------
# 1) Grab logits/probs + true labels from whatever exists
# ---------
logits = None
y_true = None

# Most common: `preds = trainer.predict(val_ds or test_ds)`
if "preds" in globals() and hasattr(preds, "predictions") and hasattr(preds, "label_ids"):
    logits = preds.predictions
    y_true = preds.label_ids

# Fallbacks (if your notebook uses different names)
elif "eval_pred" in globals() and isinstance(eval_pred, (tuple, list)) and len(eval_pred) == 2:
    logits, y_true = eval_pred

elif "test_logits" in globals() and "y_test" in globals():
    logits, y_true = test_logits, y_test

elif "val_logits" in globals() and "y_val" in globals():
    logits, y_true = val_logits, y_val

if logits is None or y_true is None:
    raise NameError(
        "❌ Can't find prediction logits + true labels.\n"
        "Make sure you have something like: preds = trainer.predict(val_ds) (or test_ds)\n"
        "and that LABEL_COLS exists."
    )

# ---------
# 2) Convert logits -> probabilities (if needed)
# ---------
logits = np.asarray(logits)
y_true = np.asarray(y_true).astype(int)

# If already probabilities (0..1), keep; else sigmoid
if logits.min() >= 0.0 and logits.max() <= 1.0:
    probs = logits
else:
    probs = _sigmoid_np(logits)

# ---------
# 3) Choose thresholds (per-label preferred, else global, else 0.5)
# ---------
if "best_per_label_t" in globals():
    thr = np.asarray(best_per_label_t, dtype=np.float32).reshape(1, -1)
elif "best_thr" in globals() and isinstance(best_thr, dict) and "LABEL_COLS" in globals():
    thr = np.asarray([best_thr[k] for k in LABEL_COLS], dtype=np.float32).reshape(1, -1)
elif "best_global_t" in globals():
    thr = float(best_global_t)
else:
    thr = 0.5

y_pred = (probs >= thr).astype(int)

# ---------
# 4) ✅ Overall accuracy (label-wise) + macro label accuracy
# ---------
overall_label_accuracy = (y_pred == y_true).mean()          # micro over all labels & samples
per_label_accuracy     = (y_pred == y_true).mean(axis=0)    # per emotion
macro_label_accuracy   = per_label_accuracy.mean()          # average across labels

print("\n==============================")
print("✅ Overall Accuracy (Label-wise, NOT subset)")
print("==============================")
print(f"Overall label-wise accuracy (micro): {overall_label_accuracy:.4f}")
print(f"Macro label accuracy (mean per label): {macro_label_accuracy:.4f}")

if "LABEL_COLS" in globals():
    print("\nPer-label accuracy:")
    for lbl, acc in zip(LABEL_COLS, per_label_accuracy):
        print(f"  {lbl:>8}: {acc:.4f}")


✅ Overall Accuracy (Label-wise, NOT subset)
Overall label-wise accuracy (micro): 0.8853
Macro label accuracy (mean per label): 0.8853

Per-label accuracy:
      Love: 0.8866
       Joy: 0.8526
     Anger: 0.8339
  Surprise: 0.9266
   Sadness: 0.8695
      Fear: 0.9398
      Hate: 0.8882
